In [ ]:
# quanti sono i file csv.gz nei labelers_log, per mese:

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter

files = sorted(
    Path(".").glob("*.csv.gz"),
    key=lambda p: int(p.name.replace(".csv.gz", ""))
)

print("n_files:", len(files))

counts = Counter()
for f in files:
    ts = int(f.name.replace(".csv.gz", ""))
    month = datetime.fromtimestamp(ts, tz=timezone.utc).strftime("%Y-%m")
    counts[month] += 1

for month, n in sorted(counts.items()):
    print(month, n)

In [ ]:
#Account-level non-neg !takedown by event ts month:
#considering manifest files include all march, plus the first file before march: 2026-01-18T22:27:22+00:00 1768775242.csv.gz

from pathlib import Path
from collections import Counter
from datetime import datetime
import csv, gzip

MANIFEST = Path("/share/storage/monade/rota/results/official_labeler_manifest_paths_mar2026.txt")

files = [Path(x.strip().split()[0]) for x in MANIFEST.read_text().splitlines() if x.strip()]

counts = Counter()

for i, path in enumerate(files, 1):
    if i % 500 == 0:
        print(f"processed {i}/{len(files)}", flush=True)

    with gzip.open(path, "rt", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("val") != "!takedown":
                continue
            if str(row.get("neg", "")).lower() == "true":
                continue

            uri = row.get("uri", "")
            if not uri.startswith("did:"):
                continue

            ts = datetime.fromisoformat(row["ts"].replace("Z", "+00:00"))
            counts[ts.strftime("%Y-%m")] += 1

print("\nAccount-level non-neg !takedown by event ts month:")
for month, n in sorted(counts.items()):
    print(month, n)

In [ ]:
#account con più di un evento di takedown positivo in marzo:

import duckdb

p = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/account_takedown_positive_mar2026_SENSITIVE.parquet"
con = duckdb.connect()

print(con.execute(f"""
SELECT
    COUNT(*) AS positivi_totali,
    SUM(CASE WHEN n_takedown_events > 1 THEN 1 ELSE 0 END) AS con_piu_di_un_takedown,
    ROUND(100.0 * SUM(CASE WHEN n_takedown_events > 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS percentuale
FROM read_parquet('{p}')
""").fetchdf().to_string(index=False))

In [ ]:
# analisi degli account con più di un takedown a marzo 2026, distribuzione del numero di eventi
#  e distanza tra primo e secondo evento

from pathlib import Path
from datetime import datetime
from collections import defaultdict
import csv, gzip
import pandas as pd

MANIFEST = Path("/share/storage/monade/rota/results/official_labeler_manifest_paths_mar2026.txt")

A = datetime.fromisoformat("2026-03-01T00:00:00+00:00")
B = datetime.fromisoformat("2026-04-01T00:00:00+00:00")

events = defaultdict(list)

files = [Path(x.strip().split()[0]) for x in MANIFEST.read_text().splitlines() if x.strip()]

for i, path in enumerate(files, 1):
    if i % 500 == 0:
        print(f"processed {i}/{len(files)}", flush=True)

    with gzip.open(path, "rt", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("val") != "!takedown":
                continue
            if str(row.get("neg", "")).lower() == "true":
                continue

            uri = row.get("uri", "")
            if not uri.startswith("did:"):
                continue

            ts = datetime.fromisoformat(row["ts"].replace("Z", "+00:00"))
            if not (A <= ts < B):
                continue

            events[uri].append(ts)

multi = {did: sorted(ts_list) for did, ts_list in events.items() if len(ts_list) > 1}

n_events = []
second_delay_hours = []
second_delay_days = []

for did, ts_list in multi.items():
    n_events.append(len(ts_list))
    delay = ts_list[1] - ts_list[0]
    second_delay_hours.append(delay.total_seconds() / 3600)
    second_delay_days.append(delay.total_seconds() / 86400)

df = pd.DataFrame({
    "n_takedown_events": n_events,
    "second_delay_hours": second_delay_hours,
    "second_delay_days": second_delay_days,
})

print("\nRISULTATI - ACCOUNT CON PIÙ DI UN TAKEDOWN IN MARZO")
print("account con >1 takedown:", len(df))

print("\nNumero di takedown per account multiplo:")
print(df["n_takedown_events"].agg(["mean", "min", "max"]).to_string())

print("\nDistanza tra primo e secondo takedown - ore:")
print(df["second_delay_hours"].agg(["mean", "min", "max"]).to_string())

print("\nDistanza tra primo e secondo takedown - giorni:")
print(df["second_delay_days"].agg(["mean", "min", "max"]).to_string())

In [ ]:
#calcolare quanti sono gli eventi account-level di takedown revocati, neg=true, in marzo 2026
#distinzione tra eventi (considera anche doppioni) e account (considera solo primo evento per account)

from pathlib import Path
from datetime import datetime
from collections import defaultdict
import csv, gzip

MANIFEST = Path("/share/storage/monade/rota/results/official_labeler_manifest_paths_mar2026.txt")

A = datetime.fromisoformat("2026-03-01T00:00:00+00:00")
B = datetime.fromisoformat("2026-04-01T00:00:00+00:00")

pos = defaultdict(list)  # neg=false
neg = defaultdict(list)  # neg=true

files = [Path(x.strip().split()[0]) for x in MANIFEST.read_text().splitlines() if x.strip()]

for path in files:
    with gzip.open(path, "rt", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("val") != "!takedown":
                continue

            uri = row.get("uri", "")
            if not uri.startswith("did:"):
                continue

            ts = datetime.fromisoformat(row["ts"].replace("Z", "+00:00"))
            if not (A <= ts < B):
                continue

            if str(row.get("neg", "")).lower() == "true":
                neg[uri].append(ts)
            else:
                pos[uri].append(ts)

n_pos_events = sum(len(v) for v in pos.values())
n_neg_events = sum(len(v) for v in neg.values())
n_total_events = n_pos_events + n_neg_events

accounts_pos = set(pos)
accounts_neg = set(neg)

accounts_with_later_revocation = 0

for did in accounts_pos:
    first_pos = min(pos[did])
    later_negs = [t for t in neg.get(did, []) if t > first_pos]
    if later_negs:
        accounts_with_later_revocation += 1

print("TAKEDOWN ACCOUNT-LEVEL - MARZO 2026")
print("eventi !takedown non-neg:", n_pos_events)
print("eventi !takedown neg=true:", n_neg_events)
print("eventi totali !takedown:", n_total_events)

print("\nFrequenza revoche su eventi:")
print(round(100 * n_neg_events / n_total_events, 4) if n_total_events else 0, "%")

print("\nAccount:")
print("account con almeno un !takedown non-neg:", len(accounts_pos))
print("account con almeno una revoca neg=true:", len(accounts_neg))
print("account positivi con revoca successiva:", accounts_with_later_revocation)

print("\nFrequenza account positivi con revoca successiva:")
print(round(100 * accounts_with_later_revocation / len(accounts_pos), 4) if accounts_pos else 0, "%")

In [ ]:
#se emessa con quanta distanza di tempo viene emessa la neg=true (mean, min, max)?

from pathlib import Path
from datetime import datetime
from collections import defaultdict
import csv, gzip
import pandas as pd

MANIFEST = Path("/share/storage/monade/rota/results/official_labeler_manifest_paths_mar2026.txt")

A = datetime.fromisoformat("2026-03-01T00:00:00+00:00")
B = datetime.fromisoformat("2026-04-01T00:00:00+00:00")

pos = defaultdict(list)
neg = defaultdict(list)

files = [Path(x.strip().split()[0]) for x in MANIFEST.read_text().splitlines() if x.strip()]

for i, path in enumerate(files, 1):
    if i % 500 == 0:
        print(f"processed {i}/{len(files)}", flush=True)

    with gzip.open(path, "rt", encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)

        for row in reader:
            if row.get("val") != "!takedown":
                continue

            uri = row.get("uri", "")
            if not uri.startswith("did:"):
                continue

            ts = datetime.fromisoformat(row["ts"].replace("Z", "+00:00"))
            if not (A <= ts < B):
                continue

            if str(row.get("neg", "")).lower() == "true":
                neg[uri].append(ts)
            else:
                pos[uri].append(ts)

delays_hours = []
delays_days = []

for did, pos_times in pos.items():
    first_td = min(pos_times)
    later_negs = [t for t in neg.get(did, []) if t > first_td]

    if later_negs:
        first_revocation = min(later_negs)
        delta = first_revocation - first_td
        delays_hours.append(delta.total_seconds() / 3600)
        delays_days.append(delta.total_seconds() / 86400)

df = pd.DataFrame({
    "delay_hours": delays_hours,
    "delay_days": delays_days
})

print("\nAccount positivi con revoca successiva:", len(df))

print("\nDistanza primo takedown → prima revoca, in ore:")
print(df["delay_hours"].agg(["mean", "min", "max"]).to_string())

print("\nDistanza primo takedown → prima revoca, in giorni:")
print(df["delay_days"].agg(["mean", "min", "max"]).to_string())

In [ ]:
#distribuzione dei positvi in marzo

import duckdb

PATH = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/account_takedown_positive_mar2026_SENSITIVE.parquet"

def show(title, df):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    print(df.to_string(index=False))

con = duckdb.connect()

con.execute(f"""
CREATE TEMP VIEW positives AS
SELECT
    did,
    TRY_CAST(takedown_at AS TIMESTAMP) AS takedown_ts,
    n_takedown_events
FROM read_parquet('{PATH}')
WHERE TRY_CAST(takedown_at AS TIMESTAMP) >= TIMESTAMP '2026-03-01'
  AND TRY_CAST(takedown_at AS TIMESTAMP) <  TIMESTAMP '2026-04-01'
""")

summary = con.execute("""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_accounts,
    MIN(takedown_ts) AS min_takedown_ts,
    MAX(takedown_ts) AS max_takedown_ts,
    SUM(n_takedown_events) AS total_takedown_events
FROM positives
""").df()

daily = con.execute("""
SELECT
    CAST(takedown_ts AS DATE) AS day,
    COUNT(*) AS n_positive_accounts,
    SUM(n_takedown_events) AS n_takedown_events
FROM positives
GROUP BY 1
ORDER BY 1
""").df()

by_10 = con.execute("""
SELECT
    CASE
        WHEN EXTRACT(DAY FROM takedown_ts) BETWEEN 1 AND 10 THEN '01-10 marzo'
        WHEN EXTRACT(DAY FROM takedown_ts) BETWEEN 11 AND 20 THEN '11-20 marzo'
        WHEN EXTRACT(DAY FROM takedown_ts) BETWEEN 21 AND 31 THEN '21-31 marzo'
    END AS bucket_10d,
    COUNT(*) AS n_positive_accounts,
    SUM(n_takedown_events) AS n_takedown_events
FROM positives
GROUP BY 1
ORDER BY MIN(takedown_ts)
""").df()

by_3 = con.execute("""
SELECT
    CONCAT(
        LPAD(CAST(1 + FLOOR((EXTRACT(DAY FROM takedown_ts) - 1) / 3) * 3 AS VARCHAR), 2, '0'),
        '-',
        LPAD(CAST(LEAST(31, 3 + FLOOR((EXTRACT(DAY FROM takedown_ts) - 1) / 3) * 3) AS VARCHAR), 2, '0'),
        ' marzo'
    ) AS bucket_3d,
    COUNT(*) AS n_positive_accounts,
    SUM(n_takedown_events) AS n_takedown_events
FROM positives
GROUP BY 1
ORDER BY MIN(takedown_ts)
""").df()

show("SUMMARY", summary)
show("DISTRIBUZIONE PER GIORNO", daily)
show("DISTRIBUZIONE PER FASCE DA 10 GIORNI", by_10)
show("DISTRIBUZIONE PER FASCE DA 3 GIORNI", by_3)

con.close()

In [ ]:
#builiding of positive datasets in 11-21 march
import duckdb
from pathlib import Path

INPUT = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/account_takedown_positive_mar2026_SENSITIVE.parquet"

OUT_DIR = Path("/share/storage/monade/rota/results/account_takedown_positive_mar2026")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = str(OUT_DIR / "positive_index_10d_strict_mar2026.parquet")

con = duckdb.connect()

con.execute(f"""
COPY (
    SELECT
        did,
        1 AS target,
        TRY_CAST(takedown_at AS TIMESTAMP) AS event_time,
        TRY_CAST(takedown_at AS TIMESTAMP) - INTERVAL 10 DAY AS window_start,
        TRY_CAST(takedown_at AS TIMESTAMP) AS window_end
    FROM read_parquet('{INPUT}')
    WHERE TRY_CAST(takedown_at AS TIMESTAMP) >= TIMESTAMP '2026-03-11'
      AND TRY_CAST(takedown_at AS TIMESTAMP) <  TIMESTAMP '2026-03-22'
    ORDER BY event_time, did
)
TO '{OUTPUT}'
(FORMAT PARQUET)
""")

print("\nCreato file:")
print(OUTPUT)

summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_accounts,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    MIN(window_start) AS min_window_start,
    MAX(window_end) AS max_window_end
FROM read_parquet('{OUTPUT}')
""").df()

print("\nSUMMARY")
print(summary.to_string(index=False))

daily = con.execute(f"""
SELECT
    CAST(event_time AS DATE) AS day,
    COUNT(*) AS n_positive_accounts
FROM read_parquet('{OUTPUT}')
GROUP BY 1
ORDER BY 1
""").df()

print("\nPOSITIVI PER GIORNO")
print(daily.to_string(index=False))

con.close()

In [ ]:
#getting the negative raw pool: just did_id and created_at of account non-positives, with created at before march 2026 and after 2020
import duckdb
from pathlib import Path

PROFILES = "/home/monade/storage/balduf/output/03-profiles.parquet"

POSITIVES = (
    "/share/storage/monade/rota/results/account_takedown_positive_mar2026/"
    "account_takedown_positive_mar2026_SENSITIVE.parquet"
)

OUT_DIR = Path("/share/storage/monade/rota/results/account_negative_mar2026")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = str(OUT_DIR / "raw_negative_pool_mar2026.parquet")

con = duckdb.connect()

print("\nCreating raw negative pool with clean created_at range...")
print(f"Profiles source:  {PROFILES}")
print(f"Positives source: {POSITIVES}")
print(f"Output:           {OUTPUT}")

con.execute(f"""
COPY (
    WITH profiles_clean AS (
        SELECT
            did_id AS did,
            MIN(TRY_CAST(created_at AS TIMESTAMP)) AS created_at
        FROM read_parquet('{PROFILES}')
        WHERE did_id IS NOT NULL
          AND TRY_CAST(created_at AS TIMESTAMP) IS NOT NULL
          AND TRY_CAST(created_at AS TIMESTAMP) >= TIMESTAMP '2020-01-01'
          AND TRY_CAST(created_at AS TIMESTAMP) <  TIMESTAMP '2026-03-01'
        GROUP BY did_id
    ),

    positives AS (
        SELECT DISTINCT did
        FROM read_parquet('{POSITIVES}')
        WHERE did IS NOT NULL
    )

    SELECT
        p.did,
        p.created_at
    FROM profiles_clean p
    LEFT JOIN positives t
        ON p.did = t.did
    WHERE t.did IS NULL
    ORDER BY p.did
)
TO '{OUTPUT}'
(FORMAT PARQUET)
""")

summary = con.execute(f"""
WITH profiles_all AS (
    SELECT
        did_id AS did,
        TRY_CAST(created_at AS TIMESTAMP) AS created_at
    FROM read_parquet('{PROFILES}')
),

profiles_clean AS (
    SELECT DISTINCT did
    FROM profiles_all
    WHERE did IS NOT NULL
      AND created_at IS NOT NULL
      AND created_at >= TIMESTAMP '2020-01-01'
      AND created_at <  TIMESTAMP '2026-03-01'
),

positive_dids AS (
    SELECT DISTINCT did
    FROM read_parquet('{POSITIVES}')
    WHERE did IS NOT NULL
),

negative_pool AS (
    SELECT *
    FROM read_parquet('{OUTPUT}')
)

SELECT
    (SELECT COUNT(*) FROM profiles_all) AS profiles_rows_total,
    (SELECT COUNT(DISTINCT did) FROM profiles_all WHERE did IS NOT NULL) AS profiles_distinct_dids_total,

    (SELECT COUNT(DISTINCT did)
     FROM profiles_all
     WHERE did IS NOT NULL
       AND created_at IS NOT NULL
       AND created_at < TIMESTAMP '2020-01-01') AS excluded_created_before_2020,

    (SELECT COUNT(DISTINCT did)
     FROM profiles_all
     WHERE did IS NOT NULL
       AND created_at IS NOT NULL
       AND created_at >= TIMESTAMP '2026-03-01') AS excluded_created_from_march_or_later,

    (SELECT COUNT(*) FROM profiles_clean) AS profiles_dids_created_2020_to_before_march_2026,
    (SELECT COUNT(*) FROM positive_dids) AS positive_dids_excluded,
    (SELECT COUNT(*) FROM negative_pool) AS raw_negative_pool_size,
    (SELECT MIN(created_at) FROM negative_pool) AS min_created_at_negative_pool,
    (SELECT MAX(created_at) FROM negative_pool) AS max_created_at_negative_pool
""").df()

print("\nSUMMARY")
print(summary.to_string(index=False))

overlap_check = con.execute(f"""
SELECT
    COUNT(*) AS overlap_negative_pool_with_positives
FROM read_parquet('{OUTPUT}') n
INNER JOIN (
    SELECT DISTINCT did
    FROM read_parquet('{POSITIVES}')
    WHERE did IS NOT NULL
) p
ON n.did = p.did
""").df()

print("\nOVERLAP CHECK")
print(overlap_check.to_string(index=False))

sample = con.execute(f"""
SELECT *
FROM read_parquet('{OUTPUT}')
LIMIT 10
""").df()

print("\nSAMPLE")
print(sample.to_string(index=False))

con.close()

In [ ]:
# building of negative_candidate_observations_by_day_10d_mar2026.parquet

import duckdb
from pathlib import Path

RAW_NEGATIVE_POOL = (
    "/share/storage/monade/rota/results/account_negative_mar2026/"
    "raw_negative_pool_mar2026.parquet"
)

OUT_DIR = Path("/share/storage/monade/rota/results/account_negative_mar2026")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = str(OUT_DIR / "negative_candidate_observations_by_day_10d_mar2026.parquet")

con = duckdb.connect()

print("\nCreating negative candidate observations by day...")
print(f"Input:  {RAW_NEGATIVE_POOL}")
print(f"Output: {OUTPUT}")

con.execute(f"""
COPY (
    WITH days AS (
        SELECT
            CAST(event_day AS TIMESTAMP) AS event_time
        FROM generate_series(
            DATE '2026-03-11',
            DATE '2026-03-21',
            INTERVAL 1 DAY
        ) AS t(event_day)
    )

    SELECT
        n.did,
        0 AS target,
        d.event_time,
        d.event_time - INTERVAL 10 DAY AS window_start,
        d.event_time AS window_end,
        n.created_at
    FROM read_parquet('{RAW_NEGATIVE_POOL}') n
    CROSS JOIN days d
    WHERE n.created_at < d.event_time - INTERVAL 10 DAY
    ORDER BY d.event_time, n.did
)
TO '{OUTPUT}'
(FORMAT PARQUET)
""")

summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_distinct_dids,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    MIN(window_start) AS min_window_start,
    MAX(window_end) AS max_window_end,
    MIN(created_at) AS min_created_at,
    MAX(created_at) AS max_created_at
FROM read_parquet('{OUTPUT}')
""").df()

print("\nSUMMARY")
print(summary.to_string(index=False))

daily = con.execute(f"""
SELECT
    CAST(event_time AS DATE) AS event_day,
    COUNT(*) AS n_negative_candidate_observations,
    COUNT(DISTINCT did) AS n_distinct_dids
FROM read_parquet('{OUTPUT}')
GROUP BY 1
ORDER BY 1
""").df()

print("\nOBSERVATIONS PER DAY")
print(daily.to_string(index=False))

sample = con.execute(f"""
SELECT *
FROM read_parquet('{OUTPUT}')
LIMIT 10
""").df()

print("\nSAMPLE")
print(sample.to_string(index=False))

con.close()

In [ ]:
#building of negative_candidate_exposure_10d_mar2026.parquet

import duckdb
from pathlib import Path

NEG_OBS = (
    "/share/storage/monade/rota/results/account_negative_mar2026/"
    "negative_candidate_observations_by_day_10d_mar2026.parquet"
)

POSTS = "/home/monade/storage/balduf/output/03-posts.parquet"
FOLLOWS = "/home/monade/storage/balduf/output/03-follows.parquet"

OUT_DIR = Path("/share/storage/monade/rota/results/account_negative_mar2026")
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT = str(OUT_DIR / "negative_candidate_exposure_10d_mar2026.parquet")

con = duckdb.connect()
con.execute("PRAGMA threads=4")

print("\nCreating negative candidate exposure dataset...")
print(f"Negative observations: {NEG_OBS}")
print(f"Posts:                 {POSTS}")
print(f"Follows:               {FOLLOWS}")
print(f"Output:                {OUTPUT}")

con.execute(f"""
COPY (
    WITH obs AS (
        SELECT
            did,
            target,
            event_time,
            window_start,
            window_end,
            created_at
        FROM read_parquet('{NEG_OBS}')
    ),

    posts_10d AS (
        SELECT
            o.did,
            o.event_time,
            COUNT(p.rkey) AS n_posts_10d
        FROM obs o
        LEFT JOIN read_parquet('{POSTS}') p
            ON p.did_id = o.did
           AND p.created_at >= o.window_start
           AND p.created_at <  o.window_end
        GROUP BY o.did, o.event_time
    ),

    follows_from_march_start AS (
        SELECT
            o.did,
            o.event_time,
            COUNT(f.rkey) AS incoming_follows_from_march_start
        FROM obs o
        LEFT JOIN read_parquet('{FOLLOWS}') f
            ON f.subject_id = o.did
           AND f.created_at >= TIMESTAMP '2026-03-01'
           AND f.created_at <  o.event_time
        GROUP BY o.did, o.event_time
    )

    SELECT
        o.did,
        o.target,
        o.event_time,
        o.window_start,
        o.window_end,
        o.created_at,
        COALESCE(p.n_posts_10d, 0) AS n_posts_10d,
        COALESCE(f.incoming_follows_from_march_start, 0) AS incoming_follows_from_march_start
    FROM obs o
    LEFT JOIN posts_10d p
        ON o.did = p.did
       AND o.event_time = p.event_time
    LEFT JOIN follows_from_march_start f
        ON o.did = f.did
       AND o.event_time = f.event_time
    ORDER BY o.event_time, o.did
)
TO '{OUTPUT}'
(FORMAT PARQUET)
""")

summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_distinct_dids,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    MIN(window_start) AS min_window_start,
    MAX(window_end) AS max_window_end,
    AVG(n_posts_10d) AS avg_n_posts_10d,
    MAX(n_posts_10d) AS max_n_posts_10d,
    AVG(incoming_follows_from_march_start) AS avg_incoming_follows_from_march_start,
    MAX(incoming_follows_from_march_start) AS max_incoming_follows_from_march_start
FROM read_parquet('{OUTPUT}')
""").df()

print("\nSUMMARY")
print(summary.to_string(index=False))

by_day = con.execute(f"""
SELECT
    CAST(event_time AS DATE) AS event_day,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_distinct_dids,
    AVG(n_posts_10d) AS avg_n_posts_10d,
    MAX(n_posts_10d) AS max_n_posts_10d,
    AVG(incoming_follows_from_march_start) AS avg_incoming_follows_from_march_start,
    MAX(incoming_follows_from_march_start) AS max_incoming_follows_from_march_start
FROM read_parquet('{OUTPUT}')
GROUP BY 1
ORDER BY 1
""").df()

print("\nSUMMARY BY DAY")
print(by_day.to_string(index=False))

sample = con.execute(f"""
SELECT *
FROM read_parquet('{OUTPUT}')
LIMIT 10
""").df()

print("\nSAMPLE")
print(sample.to_string(index=False))

con.close()

In [ ]:
#analysis of weirdness in max_posts_10d . are there duplicate in our derived dataset compared to posts.parquet?

import duckdb

POSTS = "/home/monade/storage/balduf/output/03-posts.parquet"

A = "2026-03-11 00:00:00+00:00"
B = "2026-03-21 00:00:00+00:00"

con = duckdb.connect()
con.execute("SET preserve_insertion_order=false")

print("WINDOW")
print("A =", A)
print("B =", B)

print("\nDIRECT COUNT FROM POSTS")
print(con.execute(f"""
SELECT
    COUNT(*) AS direct_rows,
    COUNT(DISTINCT did_id || ':' || rkey) AS direct_distinct_posts,
    COUNT(*) - COUNT(DISTINCT did_id || ':' || rkey) AS duplicated_rows,
    MIN(created_at) AS min_post_time,
    MAX(created_at) AS max_post_time
FROM read_parquet('{POSTS}')
WHERE created_at >= TIMESTAMPTZ '{A}'
  AND created_at <  TIMESTAMPTZ '{B}'
""").fetchdf().to_string(index=False))

print("\nTOP 30 ACCOUNTS BY DISTINCT POSTS")
print(con.execute(f"""
SELECT
    did_id,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did_id || ':' || rkey) AS n_distinct_posts,
    COUNT(*) - COUNT(DISTINCT did_id || ':' || rkey) AS duplicated_rows,
    MIN(created_at) AS first_post_at,
    MAX(created_at) AS last_post_at
FROM read_parquet('{POSTS}')
WHERE created_at >= TIMESTAMPTZ '{A}'
  AND created_at <  TIMESTAMPTZ '{B}'
GROUP BY did_id
ORDER BY n_distinct_posts DESC
LIMIT 30
""").fetchdf().to_string(index=False))

print("\nPER-ACCOUNT DISTRIBUTION")
print(con.execute(f"""
WITH per_account AS (
    SELECT
        did_id,
        COUNT(DISTINCT did_id || ':' || rkey) AS n_posts
    FROM read_parquet('{POSTS}')
    WHERE created_at >= TIMESTAMPTZ '{A}'
      AND created_at <  TIMESTAMPTZ '{B}'
    GROUP BY did_id
)
SELECT
    COUNT(*) AS n_active_accounts,
    AVG(n_posts) AS avg_posts_per_account,
    MIN(n_posts) AS min_posts_per_account,
    MAX(n_posts) AS max_posts_per_account,
    quantile_cont(n_posts, 0.50) AS p50,
    quantile_cont(n_posts, 0.90) AS p90,
    quantile_cont(n_posts, 0.99) AS p99,
    quantile_cont(n_posts, 0.999) AS p999
FROM per_account
""").fetchdf().to_string(index=False))

In [ ]:
# building of positive_exposure_10d_mar2026.parquet

import duckdb

FILE = (
    "/share/storage/monade/rota/results/account_negative_mar2026/"
    "negative_candidate_exposure_10d_mar2026.parquet"
)

con = duckdb.connect()
con.execute("PRAGMA threads=4")

print("\nChecking negative exposure dataset...")
print(f"File: {FILE}")

summary = con.execute(f"""
SELECT
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_distinct_dids,
    COUNT(*) * 1.0 / COUNT(DISTINCT did) AS avg_rows_per_did,
    MIN(event_time) AS min_event_time,
    MAX(event_time) AS max_event_time,
    COUNT(DISTINCT CAST(event_time AS DATE)) AS n_distinct_event_days
FROM read_parquet('{FILE}')
""").df()

print("\nGLOBAL SUMMARY")
print(summary.to_string(index=False))

per_did_distribution = con.execute(f"""
WITH did_counts AS (
    SELECT
        did,
        COUNT(*) AS n_rows_per_did,
        COUNT(DISTINCT CAST(event_time AS DATE)) AS n_days_per_did,
        MIN(event_time) AS min_event_time,
        MAX(event_time) AS max_event_time
    FROM read_parquet('{FILE}')
    GROUP BY did
)
SELECT
    n_rows_per_did,
    n_days_per_did,
    COUNT(*) AS n_dids
FROM did_counts
GROUP BY n_rows_per_did, n_days_per_did
ORDER BY n_rows_per_did, n_days_per_did
""").df()

print("\nROWS PER DID DISTRIBUTION")
print(per_did_distribution.to_string(index=False))

bad_dids = con.execute(f"""
WITH did_counts AS (
    SELECT
        did,
        COUNT(*) AS n_rows_per_did,
        COUNT(DISTINCT CAST(event_time AS DATE)) AS n_days_per_did,
        MIN(event_time) AS min_event_time,
        MAX(event_time) AS max_event_time
    FROM read_parquet('{FILE}')
    GROUP BY did
)
SELECT *
FROM did_counts
WHERE n_rows_per_did != 11
   OR n_days_per_did != 11
ORDER BY n_rows_per_did, did
LIMIT 50
""").df()

print("\nBAD DIDS SAMPLE")
if len(bad_dids) == 0:
    print("OK: every DID appears exactly 11 times, across 11 distinct event days.")
else:
    print(bad_dids.to_string(index=False))

by_day = con.execute(f"""
SELECT
    CAST(event_time AS DATE) AS event_day,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT did) AS n_distinct_dids
FROM read_parquet('{FILE}')
GROUP BY 1
ORDER BY 1
""").df()

print("\nROWS BY EVENT DAY")
print(by_day.to_string(index=False))

duplicate_did_day = con.execute(f"""
WITH did_day_counts AS (
    SELECT
        did,
        CAST(event_time AS DATE) AS event_day,
        COUNT(*) AS n_rows
    FROM read_parquet('{FILE}')
    GROUP BY did, CAST(event_time AS DATE)
)
SELECT *
FROM did_day_counts
WHERE n_rows != 1
ORDER BY n_rows DESC, did, event_day
LIMIT 50
""").df()

print("\nDUPLICATE DID-DAY SAMPLE")
if len(duplicate_did_day) == 0:
    print("OK: every DID has exactly one row per event day.")
else:
    print(duplicate_did_day.to_string(index=False))

con.close()


In [ ]:
#creazione del dataset che contiene i bucket indicizzati

import pandas as pd
from pathlib import Path

OUT = Path("/share/storage/monade/rota/results/exposure_bucket_definitions_mar2026_v2.parquet")

bucket_rows = [
    (0,  "0",         0,    0),
    (1,  "1",         1,    1),
    (2,  "2",         2,    2),
    (3,  "3-5",       3,    5),
    (4,  "6-10",      6,    10),
    (5,  "11-25",     11,   25),
    (6,  "26-50",     26,   50),
    (7,  "51-100",    51,   100),
    (8,  "101-250",   101,  250),
    (9,  "251-500",   251,  500),
    (10, "501-1000",  501,  1000),
    (11, "1001+",     1001, None),
]

rows = []
for variable in ["n_posts_10d", "incoming_follows_from_march_start"]:
    for bucket_index, bucket_label, lower, upper in bucket_rows:
        rows.append({
            "variable": variable,
            "bucket_index": bucket_index,
            "bucket_label": bucket_label,
            "lower_inclusive": lower,
            "upper_inclusive": upper,
        })

df = pd.DataFrame(rows)
df.to_parquet(OUT, index=False)

print(f"Wrote: {OUT}")
print(df.to_string(index=False))

In [ ]:
#creazione dei dataset did level positivi e negativi con bucket di esposizione

import duckdb
from pathlib import Path

POS_IN = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/positive_exposure_10d_mar2026.parquet"
NEG_IN = "/share/storage/monade/rota/results/account_negative_mar2026/negative_candidate_exposure_10d_mar2026.parquet"

POS_OUT = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/positive_exposure_10d_mar2026_bucketed_v2.parquet"
NEG_OUT = "/share/storage/monade/rota/results/account_negative_mar2026/negative_candidate_exposure_10d_mar2026_bucketed_v2.parquet"

con = duckdb.connect()

bucket_case_posts = """
CASE
    WHEN n_posts_10d = 0 THEN 0
    WHEN n_posts_10d = 1 THEN 1
    WHEN n_posts_10d = 2 THEN 2
    WHEN n_posts_10d BETWEEN 3 AND 5 THEN 3
    WHEN n_posts_10d BETWEEN 6 AND 10 THEN 4
    WHEN n_posts_10d BETWEEN 11 AND 25 THEN 5
    WHEN n_posts_10d BETWEEN 26 AND 50 THEN 6
    WHEN n_posts_10d BETWEEN 51 AND 100 THEN 7
    WHEN n_posts_10d BETWEEN 101 AND 250 THEN 8
    WHEN n_posts_10d BETWEEN 251 AND 500 THEN 9
    WHEN n_posts_10d BETWEEN 501 AND 1000 THEN 10
    WHEN n_posts_10d >= 1001 THEN 11
    ELSE NULL
END
"""

bucket_case_follows = """
CASE
    WHEN incoming_follows_from_march_start = 0 THEN 0
    WHEN incoming_follows_from_march_start = 1 THEN 1
    WHEN incoming_follows_from_march_start = 2 THEN 2
    WHEN incoming_follows_from_march_start BETWEEN 3 AND 5 THEN 3
    WHEN incoming_follows_from_march_start BETWEEN 6 AND 10 THEN 4
    WHEN incoming_follows_from_march_start BETWEEN 11 AND 25 THEN 5
    WHEN incoming_follows_from_march_start BETWEEN 26 AND 50 THEN 6
    WHEN incoming_follows_from_march_start BETWEEN 51 AND 100 THEN 7
    WHEN incoming_follows_from_march_start BETWEEN 101 AND 250 THEN 8
    WHEN incoming_follows_from_march_start BETWEEN 251 AND 500 THEN 9
    WHEN incoming_follows_from_march_start BETWEEN 501 AND 1000 THEN 10
    WHEN incoming_follows_from_march_start >= 1001 THEN 11
    ELSE NULL
END
"""

for in_path, out_path in [(POS_IN, POS_OUT), (NEG_IN, NEG_OUT)]:
    query = f"""
    COPY (
        WITH base AS (
            SELECT
                *,
                {bucket_case_posts} AS post_bucket_index,
                {bucket_case_follows} AS follow_bucket_index
            FROM read_parquet('{in_path}')
        )
        SELECT
            *,
            post_bucket_index * 100 + follow_bucket_index AS exposure_bucket_index
        FROM base
    )
    TO '{out_path}'
    (FORMAT PARQUET)
    """
    con.execute(query)
    print(f"Wrote: {out_path}")

print("\nValidation:")
for path in [POS_OUT, NEG_OUT]:
    df = con.execute(f"""
        SELECT
            COUNT(*) AS n_rows,
            COUNT(DISTINCT did) AS n_dids,
            COUNT(DISTINCT exposure_bucket_index) AS n_exposure_buckets,
            MIN(post_bucket_index) AS min_post_bucket,
            MAX(post_bucket_index) AS max_post_bucket,
            MIN(follow_bucket_index) AS min_follow_bucket,
            MAX(follow_bucket_index) AS max_follow_bucket
        FROM read_parquet('{path}')
    """).df()
    print("\n", path)
    print(df.to_string(index=False))

con.close()

In [ ]:
#causa refuso su incoming_follower, ho rifatto l exposure corretta e riassegnato i bucket.
# incoming follower si basa sempre su 10 giorni
'
from pathlib import Path
import duckdb

# ============================================================
# PATHS
# ============================================================

FOLLOWS_PATH = "/home/monade/storage/balduf/output/03-follows.parquet"

POS_IN = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/positive_exposure_10d_mar2026.parquet"
NEG_IN = "/share/storage/monade/rota/results/account_negative_mar2026/negative_candidate_exposure_10d_mar2026.parquet"

POS_OUT = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/positive_exposure_10d_mar2026_v3.parquet"
NEG_OUT = "/share/storage/monade/rota/results/account_negative_mar2026/negative_candidate_exposure_10d_mar2026_v3.parquet"

POS_BUCKETED_OUT = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/positive_exposure_10d_mar2026_bucketed_v3.parquet"
NEG_BUCKETED_OUT = "/share/storage/monade/rota/results/account_negative_mar2026/negative_candidate_exposure_10d_mar2026_bucketed_v3.parquet"


# ============================================================
# BUCKET CASE EXPRESSION
# ============================================================

def bucket_expr(col: str) -> str:
    return f"""
        CASE
            WHEN {col} = 0 THEN 0
            WHEN {col} = 1 THEN 1
            WHEN {col} = 2 THEN 2
            WHEN {col} BETWEEN 3 AND 5 THEN 3
            WHEN {col} BETWEEN 6 AND 10 THEN 4
            WHEN {col} BETWEEN 11 AND 25 THEN 5
            WHEN {col} BETWEEN 26 AND 50 THEN 6
            WHEN {col} BETWEEN 51 AND 100 THEN 7
            WHEN {col} BETWEEN 101 AND 250 THEN 8
            WHEN {col} BETWEEN 251 AND 500 THEN 9
            WHEN {col} BETWEEN 501 AND 1000 THEN 10
            WHEN {col} >= 1001 THEN 11
            ELSE NULL
        END
    """


# ============================================================
# CORE FUNCTIONS
# ============================================================

def create_v3_exposure(con: duckdb.DuckDBPyConnection, input_path: str, output_path: str, label: str) -> None:
    print(f"\n=== Creating v3 exposure dataset: {label} ===")
    print(f"Input : {input_path}")
    print(f"Output: {output_path}")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE base AS
        SELECT *
        FROM read_parquet('{input_path}')
    """)

    cols = con.execute("DESCRIBE base").fetchdf()["column_name"].tolist()

    required_cols = {"did", "event_time", "window_start", "window_end", "n_posts_10d"}
    missing = required_cols - set(cols)
    if missing:
        raise ValueError(f"{label}: missing required columns: {sorted(missing)}")

    # Drop old temporally-biased metric and any old bucket columns if present.
    select_cols = [
        c for c in cols
        if c not in {
            "incoming_follows_from_march_start",
            "incoming_follows_10d",
            "post_bucket_index",
            "follow_bucket_index",
            "follow_10d_bucket_index",
            "exposure_bucket_index",
        }
    ]

    select_cols_sql = ",\n                ".join([f"b.{c}" for c in select_cols])

    con.execute(f"""
        CREATE OR REPLACE TEMP TABLE follow_counts AS
        SELECT
            b.did,
            b.event_time,
            COUNT(f.did_id) AS incoming_follows_10d
        FROM base b
        LEFT JOIN read_parquet('{FOLLOWS_PATH}') f
            ON f.subject_id = b.did
           AND f.created_at >= b.window_start
           AND f.created_at <  b.window_end
        GROUP BY
            b.did,
            b.event_time
    """)

    con.execute(f"""
        COPY (
            SELECT
                {select_cols_sql},
                COALESCE(fc.incoming_follows_10d, 0)::BIGINT AS incoming_follows_10d
            FROM base b
            LEFT JOIN follow_counts fc
                ON fc.did = b.did
               AND fc.event_time = b.event_time
        )
        TO '{output_path}'
        (FORMAT PARQUET)
    """)

    n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
    n_dids = con.execute(f"SELECT COUNT(DISTINCT did) FROM read_parquet('{output_path}')").fetchone()[0]

    stats = con.execute(f"""
        SELECT
            MIN(incoming_follows_10d) AS min_incoming_follows_10d,
            AVG(incoming_follows_10d) AS avg_incoming_follows_10d,
            MAX(incoming_follows_10d) AS max_incoming_follows_10d
        FROM read_parquet('{output_path}')
    """).fetchdf()

    print(f"Rows         : {n_rows:,}")
    print(f"Distinct DID : {n_dids:,}")
    print(stats.to_string(index=False))


def create_bucketed(con: duckdb.DuckDBPyConnection, input_path: str, output_path: str, label: str) -> None:
    print(f"\n=== Creating bucketed v3 dataset: {label} ===")
    print(f"Input : {input_path}")
    print(f"Output: {output_path}")

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    con.execute(f"""
        COPY (
            WITH bucketed AS (
                SELECT
                    *,
                    {bucket_expr("n_posts_10d")} AS post_bucket_index,
                    {bucket_expr("incoming_follows_10d")} AS follow_10d_bucket_index
                FROM read_parquet('{input_path}')
            )
            SELECT
                *,
                post_bucket_index * 100 + follow_10d_bucket_index AS exposure_bucket_index
            FROM bucketed
        )
        TO '{output_path}'
        (FORMAT PARQUET)
    """)

    n_rows = con.execute(f"SELECT COUNT(*) FROM read_parquet('{output_path}')").fetchone()[0]
    n_buckets = con.execute(f"""
        SELECT COUNT(DISTINCT exposure_bucket_index)
        FROM read_parquet('{output_path}')
    """).fetchone()[0]

    bucket_summary = con.execute(f"""
        SELECT
            post_bucket_index,
            follow_10d_bucket_index,
            exposure_bucket_index,
            COUNT(*) AS n_rows
        FROM read_parquet('{output_path}')
        GROUP BY 1, 2, 3
        ORDER BY 1, 2
        LIMIT 30
    """).fetchdf()

    print(f"Rows             : {n_rows:,}")
    print(f"Exposure buckets : {n_buckets:,}")
    print("\nFirst 30 bucket cells:")
    print(bucket_summary.to_string(index=False))


def check_output(con: duckdb.DuckDBPyConnection, path: str, label: str) -> None:
    print(f"\n=== Final check: {label} ===")

    df = con.execute(f"""
        SELECT
            COUNT(*) AS n_rows,
            COUNT(DISTINCT did) AS n_distinct_dids,
            MIN(event_time) AS min_event_time,
            MAX(event_time) AS max_event_time,
            MIN(n_posts_10d) AS min_posts_10d,
            MAX(n_posts_10d) AS max_posts_10d,
            MIN(incoming_follows_10d) AS min_incoming_follows_10d,
            MAX(incoming_follows_10d) AS max_incoming_follows_10d,
            COUNT(*) FILTER (WHERE post_bucket_index IS NULL) AS null_post_bucket,
            COUNT(*) FILTER (WHERE follow_10d_bucket_index IS NULL) AS null_follow_bucket,
            COUNT(*) FILTER (WHERE exposure_bucket_index IS NULL) AS null_exposure_bucket
        FROM read_parquet('{path}')
    """).fetchdf()

    print(df.to_string(index=False))


# ============================================================
# MAIN
# ============================================================

def main() -> None:
    con = duckdb.connect()

    con.execute("PRAGMA threads=8")
    con.execute("PRAGMA memory_limit='16GB'")

    create_v3_exposure(
        con=con,
        input_path=POS_IN,
        output_path=POS_OUT,
        label="positive",
    )

    create_v3_exposure(
        con=con,
        input_path=NEG_IN,
        output_path=NEG_OUT,
        label="negative",
    )

    create_bucketed(
        con=con,
        input_path=POS_OUT,
        output_path=POS_BUCKETED_OUT,
        label="positive",
    )

    create_bucketed(
        con=con,
        input_path=NEG_OUT,
        output_path=NEG_BUCKETED_OUT,
        label="negative",
    )

    check_output(con, POS_BUCKETED_OUT, "positive bucketed v3")
    check_output(con, NEG_BUCKETED_OUT, "negative bucketed v3")

    print("\nDONE.")
    print(f"Positive v3         : {POS_OUT}")
    print(f"Negative v3         : {NEG_OUT}")
    print(f"Positive bucketed v3: {POS_BUCKETED_OUT}")
    print(f"Negative bucketed v3: {NEG_BUCKETED_OUT}")


if __name__ == "__main__":
    main()

In [ ]:
#controllo che i bucket non siano vuoti

python <<'PY'
import duckdb
import pandas as pd

POS_PATH = "/share/storage/monade/rota/results/account_takedown_positive_mar2026/positive_exposure_10d_mar2026_bucketed_v3.parquet"
NEG_PATH = "/share/storage/monade/rota/results/account_negative_mar2026/negative_candidate_exposure_10d_mar2026_bucketed_v3.parquet"
BUCKET_DEF_PATH = "/share/storage/monade/rota/results/exposure_bucket_definitions_mar2026_v2.parquet"

con = duckdb.connect()

query = f"""
WITH pos AS (
    SELECT
        CAST(event_time AS DATE) AS event_day,
        exposure_bucket_index,
        COUNT(*) AS n_pos
    FROM read_parquet('{POS_PATH}')
    GROUP BY 1, 2
),
neg AS (
    SELECT
        CAST(event_time AS DATE) AS event_day,
        exposure_bucket_index,
        COUNT(*) AS n_neg
    FROM read_parquet('{NEG_PATH}')
    GROUP BY 1, 2
),
all_buckets AS (
    SELECT event_day, exposure_bucket_index FROM pos
    UNION
    SELECT event_day, exposure_bucket_index FROM neg
)
SELECT
    a.event_day,
    a.exposure_bucket_index,
    CAST(FLOOR(a.exposure_bucket_index / 100) AS INTEGER) AS post_bucket_index,
    CAST(a.exposure_bucket_index % 100 AS INTEGER) AS follow_bucket_index,
    COALESCE(p.n_pos, 0) AS n_pos,
    COALESCE(n.n_neg, 0) AS n_neg,
    CASE
        WHEN COALESCE(n.n_neg, 0) = 0 THEN NULL
        ELSE ROUND(COALESCE(p.n_pos, 0)::DOUBLE / n.n_neg, 6)
    END AS pos_neg_ratio
FROM all_buckets a
LEFT JOIN pos p
    ON a.event_day = p.event_day
   AND a.exposure_bucket_index = p.exposure_bucket_index
LEFT JOIN neg n
    ON a.event_day = n.event_day
   AND a.exposure_bucket_index = n.exposure_bucket_index
ORDER BY
    a.event_day,
    a.exposure_bucket_index
"""

df = con.execute(query).df()

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("\n=== POSITIVI E NEGATIVI PER EVENT_DAY × EXPOSURE_BUCKET ===\n")
print(df.to_string(index=False))

print("\n=== SUMMARY GENERALE ===\n")
summary = con.execute(f"""
WITH pos AS (
    SELECT COUNT(*) AS n_pos
    FROM read_parquet('{POS_PATH}')
),
neg AS (
    SELECT COUNT(*) AS n_neg
    FROM read_parquet('{NEG_PATH}')
),
pos_buckets AS (
    SELECT COUNT(DISTINCT exposure_bucket_index) AS n_pos_buckets
    FROM read_parquet('{POS_PATH}')
),
neg_buckets AS (
    SELECT COUNT(DISTINCT exposure_bucket_index) AS n_neg_buckets
    FROM read_parquet('{NEG_PATH}')
)
SELECT
    pos.n_pos,
    neg.n_neg,
    pos_buckets.n_pos_buckets,
    neg_buckets.n_neg_buckets
FROM pos, neg, pos_buckets, neg_buckets
""").df()

print(summary.to_string(index=False))

print("\n=== BUCKET CON POSITIVI MA ZERO NEGATIVI ===\n")
problem = df[(df["n_pos"] > 0) & (df["n_neg"] == 0)]
if len(problem) == 0:
    print("Nessun bucket problematico: ogni bucket con positivi ha almeno un negativo nello stesso giorno.")
else:
    print(problem.to_string(index=False))

print("\n=== BUCKET CON POSITIVI E POCHI NEGATIVI (< 20) ===\n")
thin = df[(df["n_pos"] > 0) & (df["n_neg"] > 0) & (df["n_neg"] < 20)]
if len(thin) == 0:
    print("Nessun bucket sottile con meno di 20 negativi.")
else:
    print(thin.to_string(index=False))

con.close()
PY